# Neural-network inversion of the Lyα forest

## From a local analytic inverse to learned non-local reconstruction

This notebook compares three reconstructions of the one-dimensional gas-density
field:

1. the traditional local FGPA inverse;
2. a convolutional encoder–decoder without skip connections; and
3. a depth-matched one-dimensional U-Net with skip connections.

The neural construction is gradual. We first understand convolution,
pooling, the bottleneck, and decoding. We then add only the U-Net skip connections,
keeping the remaining layers and training procedure fixed.

The methods are compared both field by field and through the density PDF, 1D power
spectrum, and a specified 1D bispectrum configuration.

## 1. Requirements, imports, and reproducibility

Required packages are `numpy`, `scipy`, `matplotlib`, and `torch`. The default
model is intentionally small and runs on CPU. Fixed random seeds make the split,
noise fields, initialization, and mini-batch order reproducible.

In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LogNorm
from scipy import ndimage, special
from scipy.optimize import brentq

import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(7)
torch.manual_seed(7)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(min(4, max(1, torch.get_num_threads())))
device = torch.device("cpu")

plt.rcParams.update({
    "figure.figsize": (11, 5),
    "figure.dpi": 120,
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "legend.frameon": False,
})

BLACK = "black"
BLUE = "royalblue"
ORANGE = "darkorange"
PURPLE = "slateblue"
RED = "crimson"
GREY = "grey"

print(f"PyTorch {torch.__version__}; device = {device}")
print(f"CPU threads used by PyTorch = {torch.get_num_threads()}")

## 2. Configuration

The defaults balance physical fidelity and laptop runtime. Increasing the number
of training skewers or epochs is an experiment, not a requirement for understanding
the workflow.

In [ ]:
# Data and cosmology
data_directory = Path("Sims/CMD_z=2_grid128")
unet_checkpoint_path = Path("trained_models/lya_forest_unet_density.pt")
box_size = 25.0              # h^-1 cMpc
box_redshift = 2.0
h_camels = 0.6711
Omega_m = 0.30

# Median temperature-density fit used by FGPA
eos_delta_min = 0.10
eos_delta_max = 2.00
eos_number_of_bins = 20

# Mock observation shared by the two inverse methods
signal_to_noise = 30.0
instrument_fwhm_kms = 50.0
minimum_flux_floor = 0.0

# Neural-network architecture
# Change these two values to alter both the encoder and the mirrored decoder.
number_of_encoder_decoder_layers = 2
convolution_kernel_size = 5       # use an odd integer: 1, 3, 5, 7, ...
base_channels = 8
# Dataset size and training budget
training_skewers_per_box = 96
validation_skewers_per_box = 64
test_skewers_per_box = 16
number_of_epochs = 100
batch_size = 64
learning_rate = 1.0e-3

# Fair comparison scale
comparison_fwhm = 1.0        # h^-1 cMpc

## 3. Load fields and define leakage-safe splits

We reconstruct

$$
\Delta_b(x)=\frac{\rho_b(x)}{\bar\rho_b}.
$$

The mean $\bar\rho_b$ is measured from each complete three-dimensional box. We
split by simulation, rather than randomly mixing skewers from every simulation:

| Simulations | Purpose | Used to update network weights? |
|---|---|---|
| 0–17 | training | yes |
| 18–19 | validation and model selection | no gradient updates |
| 20–26 | final test | never |

This is stricter than a random skewer split because all sightlines from a test
realization remain unseen during development.

In [ ]:
gas_path = data_directory / "Grids_Mgas_IllustrisTNG_CV_128_z=2.0.npy"
hi_path = data_directory / "Grids_HI_IllustrisTNG_CV_128_z=2.0.npy"
temperature_path = data_directory / "Grids_T_IllustrisTNG_CV_128_z=2.0.npy"

missing_paths = [
    path for path in (gas_path, hi_path, temperature_path) if not path.exists()
]
if missing_paths:
    missing_names = "\n".join(f"  - {path}" for path in missing_paths)
    raise FileNotFoundError(
        "The CAMELS grids are required for this notebook. Missing:\n"
        + missing_names
    )

gas_boxes = np.load(gas_path, mmap_mode="r")
hi_boxes = np.load(hi_path, mmap_mode="r")
temperature_boxes = np.load(temperature_path, mmap_mode="r")

number_of_cells = gas_boxes.shape[1]
cell_size = box_size / number_of_cells
x = (np.arange(number_of_cells) + 0.5) * cell_size

training_simulations = np.arange(0, 18)
validation_simulations = np.arange(18, 20)
test_simulations = np.arange(20, 27)

assert gas_boxes.shape[0] >= 27
assert hi_boxes.shape == gas_boxes.shape
assert temperature_boxes.shape == gas_boxes.shape

gas_means = np.array([
    gas_boxes[index].mean(dtype=np.float64)
    for index in range(gas_boxes.shape[0])
])

print(f"Grid shape: {gas_boxes.shape}")
print(f"Cell width: {cell_size:.3f} h^-1 cMpc")
print(f"Training / validation / test simulations: "
      f"{len(training_simulations)} / {len(validation_simulations)} / "
      f"{len(test_simulations)}")

## 4. Fit the training-set temperature–density relation

The diffuse photoionized IGM is approximately described by

$$T(\Delta_b)=T_0\Delta_b^{\gamma-1}.$$

We sample gas cells only from the training simulations, construct a logarithmic
temperature–density phase diagram, compute the median $T$ in logarithmic
$\Delta_b$ bins, and fit the medians over $0.1\leq\Delta_b\leq2$.

The fit is used only by the FGPA baseline. The neural network is not given
$T_0$ or $\gamma$; it must learn the statistical flux–density relation from its
paired training examples.

In [ ]:
phase_rng = np.random.default_rng(41)
phase_delta_parts = []
phase_temperature_parts = []

cells_sampled_per_box = 100_000
for simulation in training_simulations:
    gas_flat = np.asarray(gas_boxes[simulation]).ravel()
    temperature_flat = np.asarray(temperature_boxes[simulation]).ravel()
    sample_size = min(cells_sampled_per_box, gas_flat.size)
    sample = phase_rng.choice(gas_flat.size, sample_size, replace=False)

    phase_delta_parts.append(
        gas_flat[sample].astype(float) / gas_means[simulation]
    )
    phase_temperature_parts.append(
        temperature_flat[sample].astype(float)
    )

phase_delta = np.concatenate(phase_delta_parts)
phase_temperature = np.concatenate(phase_temperature_parts)
finite_phase = (
    np.isfinite(phase_delta)
    & np.isfinite(phase_temperature)
    & (phase_delta > 0.0)
    & (phase_temperature > 0.0)
)
phase_delta = phase_delta[finite_phase]
phase_temperature = phase_temperature[finite_phase]

median_edges = np.logspace(np.log10(0.05), np.log10(10.0),
                           eos_number_of_bins + 1)
median_centres = np.sqrt(median_edges[:-1] * median_edges[1:])
median_temperature = np.full(eos_number_of_bins, np.nan)
cells_per_bin = np.zeros(eos_number_of_bins, dtype=int)

bin_number = np.digitize(phase_delta, median_edges) - 1
for index in range(eos_number_of_bins):
    in_bin = bin_number == index
    cells_per_bin[index] = in_bin.sum()
    if cells_per_bin[index] > 0:
        median_temperature[index] = np.median(phase_temperature[in_bin])

fit_bins = (
    np.isfinite(median_temperature)
    & (cells_per_bin >= 100)
    & (median_centres >= eos_delta_min)
    & (median_centres <= eos_delta_max)
)

gamma_minus_one, log10_T0 = np.polyfit(
    np.log10(median_centres[fit_bins]),
    np.log10(median_temperature[fit_bins]),
    1,
)
inversion_T0 = 10.0**log10_T0
inversion_gamma = 1.0 + gamma_minus_one
inversion_beta = 2.0 - 0.7 * (inversion_gamma - 1.0)

print(f"Training-set thermal fit: T0 = {inversion_T0:,.0f} K")
print(f"gamma = {inversion_gamma:.3f}; beta = {inversion_beta:.3f}")

In [ ]:
density_edges = np.logspace(-3.0, 3.0, 110)
temperature_edges = np.logspace(2.0, 8.0, 110)

fig, ax = plt.subplots(figsize=(8.5, 6.2), constrained_layout=True)
image = ax.hist2d(
    phase_delta,
    phase_temperature,
    bins=(density_edges, temperature_edges),
    norm=LogNorm(),
    cmap="magma",
    cmin=1,
)
colourbar = fig.colorbar(image[3], ax=ax, pad=0.02)
colourbar.set_label("number of sampled training cells")

valid_medians = np.isfinite(median_temperature)
ax.plot(median_centres[valid_medians], median_temperature[valid_medians],
        color="white", marker="o", ms=4, lw=1.6,
        markeredgecolor=BLACK, markeredgewidth=0.4,
        label="median temperature")

fitted_delta = np.logspace(np.log10(eos_delta_min),
                           np.log10(eos_delta_max), 100)
fitted_temperature = inversion_T0 * fitted_delta**(inversion_gamma - 1.0)
ax.plot(fitted_delta, fitted_temperature, color="turquoise", lw=3,
        label=rf"fit: $T_0={inversion_T0/1000:.1f}$ kK, "
              rf"$\gamma={inversion_gamma:.2f}$")
ax.axvspan(eos_delta_min, eos_delta_max, color="turquoise", alpha=0.09,
           label="fitted range")
ax.set(xscale="log", yscale="log",
       xlabel=r"gas overdensity $\Delta_b$",
       ylabel=r"temperature $T$ [K]",
       title="Training-set temperature–density phase diagram")
ax.legend(loc="lower right")
plt.show()

## 5. Forward model

The spectrum is generated from the simulated H I number density and temperature.
For absorber cell $i$,

$$b_i=\sqrt{\frac{2k_{\rm B}T_i}{m_{\rm H}}},$$

and its neutral-hydrogen column contributes a Voigt profile to neighbouring
spectral pixels. The optical depth is therefore non-local:

$$\tau(v_j)=\sum_i \tau_i(v_j).$$

We then compute $F=e^{-\tau}$, convolve the flux with a Gaussian line-spread
function, and add Gaussian pixel noise.

Hubble-flow mapping and thermal velocities are included. A signed gas peculiar-
velocity grid is not present in these arrays, so coherent line-of-sight peculiar
velocities are omitted and this limitation is stated explicitly.

In [ ]:
# Physical constants in cgs units
MSUN_G = 1.98847e33
MPC_CM = 3.085677581e24
M_H_G = 1.6735575e-24
K_B = 1.380649e-16
C_CMS = 2.99792458e10
C_KMS = C_CMS / 1.0e5

# Ly-alpha atomic constants
LAMBDA_ALPHA_CM = 1215.67e-8
GAMMA_ALPHA = 6.262e8
I_ALPHA = 4.45e-18

H_z = 100.0 * h_camels * np.sqrt(
    Omega_m * (1.0 + box_redshift)**3 + (1.0 - Omega_m)
)
distance_from_centre_Mpc = (x - box_size / 2.0) / h_camels
absorber_redshift = (
    box_redshift + H_z * distance_from_centre_Mpc / C_KMS
)


def hi_grid_to_number_density(rho_hi_grid):
    '''Convert a CMD H I grid to physical H I number density in cm^-3.'''
    rho_comoving = rho_hi_grid * MSUN_G * h_camels**2 / MPC_CM**3
    rho_physical = rho_comoving * (1.0 + absorber_redshift)**3
    return rho_physical / M_H_G


def continuous_voigt_optical_depth(rho_hi_grid, temperature):
    '''Optical depth from cell-sampled H I and temperature fields.'''
    n_hi = hi_grid_to_number_density(rho_hi_grid)
    nu_alpha = C_CMS / LAMBDA_ALPHA_CM
    cell_width_cm = cell_size * MPC_CM / h_camels

    safe_temperature = np.clip(temperature, 10.0, None)
    b_cms = np.sqrt(2.0 * K_B * safe_temperature / M_H_G)
    damping = GAMMA_ALPHA * C_CMS / (4.0 * np.pi * nu_alpha * b_cms)

    velocity_offset = (
        C_CMS
        * (absorber_redshift[:, None] - absorber_redshift[None, :])
        / (1.0 + absorber_redshift[None, :])
    )
    profile = np.real(special.wofz(
        velocity_offset / b_cms[:, None] + 1j * damping[:, None]
    ))
    amplitude = (
        C_CMS * I_ALPHA * cell_width_cm * n_hi
        / (np.sqrt(np.pi) * b_cms * (1.0 + absorber_redshift))
    )
    return np.sum(amplitude[:, None] * profile, axis=0)


velocity_pixel_width = H_z * (cell_size / h_camels) / (1.0 + box_redshift)
instrument_sigma_pixels = (
    instrument_fwhm_kms
    / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    / velocity_pixel_width
)
noise_sigma = 1.0 / signal_to_noise
flux_floor = max(minimum_flux_floor, 2.0 * noise_sigma)


def observe_skewer(rho_hi, temperature, noise_rng):
    '''Return intrinsic, instrumentally smoothed, and noisy flux.'''
    optical_depth = continuous_voigt_optical_depth(rho_hi, temperature)
    intrinsic_flux = np.exp(-optical_depth)
    instrumental_flux = ndimage.gaussian_filter1d(
        intrinsic_flux, instrument_sigma_pixels, mode="wrap"
    )
    observed_flux = instrumental_flux + noise_rng.normal(
        0.0, noise_sigma, number_of_cells
    )
    return intrinsic_flux, instrumental_flux, observed_flux


print(f"Instrument FWHM = {instrument_fwhm_kms:.1f} km/s")
print(f"Instrument sigma = {instrument_sigma_pixels:.2f} pixels")
print(f"S/N = {signal_to_noise:.1f}; sigma_F = {noise_sigma:.4f}")
print(f"Two-sigma saturation floor = {flux_floor:.4f}")

## 6. Generate paired examples

Each pair consists of

$$
\underbrace{F_{\rm obs}(x)}_{\text{network input}}
\quad\longrightarrow\quad
\underbrace{\ln\Delta_b(x)}_{\text{supervised target}}.
$$

Predicting log density reduces dynamic range and guarantees a positive density
after exponentiation. The network never receives the target H I or temperature
fields as input; those fields are used only to produce realistic flux.

The central $(y,z)=(64,64)$ skewer is included as the first example from every
test simulation, making the representative plot deterministic.

In [ ]:
def choose_positions(number, rng, include_centre=False):
    '''Choose unique transverse coordinates in one simulation box.'''
    selected = []
    if include_centre:
        selected.append((number_of_cells // 2, number_of_cells // 2))

    centre_flat = (number_of_cells // 2) * number_of_cells + number_of_cells // 2
    available = np.delete(np.arange(number_of_cells**2), centre_flat)
    random_flat = rng.choice(
        available, number - len(selected), replace=False
    )
    selected.extend(divmod(int(index), number_of_cells) for index in random_flat)
    return selected


def build_dataset(simulations, skewers_per_box, seed, include_centre=False):
    '''Generate density targets and realistic flux for a simulation split.'''
    position_rng = np.random.default_rng(seed)
    noise_rng = np.random.default_rng(seed + 1)

    densities = []
    intrinsic_fluxes = []
    instrumental_fluxes = []
    observed_fluxes = []
    simulation_labels = []

    for simulation in simulations:
        positions = choose_positions(
            skewers_per_box, position_rng, include_centre=include_centre
        )
        for y_index, z_index in positions:
            density = (
                gas_boxes[simulation, :, y_index, z_index].astype(float)
                / gas_means[simulation]
            )
            rho_hi = hi_boxes[
                simulation, :, y_index, z_index
            ].astype(float)
            temperature = temperature_boxes[
                simulation, :, y_index, z_index
            ].astype(float)

            intrinsic, instrumental, observed = observe_skewer(
                rho_hi, temperature, noise_rng
            )
            densities.append(density)
            intrinsic_fluxes.append(intrinsic)
            instrumental_fluxes.append(instrumental)
            observed_fluxes.append(observed)
            simulation_labels.append(simulation)

    return {
        "density": np.asarray(densities, dtype=np.float32),
        "intrinsic_flux": np.asarray(intrinsic_fluxes, dtype=np.float32),
        "instrumental_flux": np.asarray(instrumental_fluxes, dtype=np.float32),
        "observed_flux": np.asarray(observed_fluxes, dtype=np.float32),
        "simulation": np.asarray(simulation_labels),
    }

In [ ]:
generation_start = time.perf_counter()

train_data = build_dataset(
    training_simulations, training_skewers_per_box, seed=101
)
validation_data = build_dataset(
    validation_simulations, validation_skewers_per_box, seed=202
)
test_data = build_dataset(
    test_simulations, test_skewers_per_box, seed=303, include_centre=True
)

print(f"Training pairs:   {train_data['density'].shape}")
print(f"Validation pairs: {validation_data['density'].shape}")
print(f"Test pairs:       {test_data['density'].shape}")
print(f"Pair generation time: {time.perf_counter()-generation_start:.1f} s")

In [ ]:
representative = 0
representative_simulation = int(test_data["simulation"][representative])

fig, axes = plt.subplots(
    2, 1, figsize=(11, 6.4), sharex=True, constrained_layout=True
)

axes[0].plot(
    x, test_data["intrinsic_flux"][representative],
    color=BLACK, lw=1.7, label="intrinsic Voigt flux"
)
axes[0].plot(
    x, test_data["instrumental_flux"][representative],
    color=BLUE, lw=2.0, label="after instrument"
)
axes[0].step(
    x, test_data["observed_flux"][representative], where="mid",
    color=ORANGE, lw=1.0, alpha=0.95, label="noisy observed spectrum"
)
axes[0].axhline(0.0, color=GREY, lw=0.7)
axes[0].axhline(1.0, color=GREY, lw=0.7, ls=":")
axes[0].set_ylim(-0.12, 1.12)
axes[0].set_ylabel("transmitted flux")
axes[0].set_title(
    f"Held-out simulation {representative_simulation}: realistic mock spectrum"
)
axes[0].legend(ncol=3)

axes[1].semilogy(
    x, test_data["density"][representative], color=BLACK, lw=2.0
)
axes[1].set_ylabel(r"true gas $\Delta_b$")
axes[1].set_xlabel(r"distance $x$ [$h^{-1}$ cMpc]")
plt.show()

### Forward-model audit

The gas density in the lower panel is a hidden target. It did not generate the
spectrum through FGPA. The upper panel was produced from the corresponding
simulated H I and temperature fields using the Voigt calculation.

The distinction prevents a circular test: if one generated
$F=\exp[-A\Delta_b^\beta]$ and inverted the same formula, FGPA would succeed by
construction and the comparison would say little about real Lyα inversion.

## 7. Traditional FGPA baseline

Combining photoionization equilibrium with the fitted thermal relation gives

$$
\tau_{\rm FGPA}=A
\left(\frac{T_0}{10^4\,{\rm K}}\right)^{-0.7}
\Delta_b^\beta,
\qquad \beta=2-0.7(\gamma-1).
$$

We calibrate the effective amplitude $A$ by matching the mean intrinsic flux of
the realistic **training** skewers. The held-out test flux and density play no role
in this calibration.

In [ ]:
temperature_factor = (inversion_T0 / 1.0e4)**(-0.7)
calibration_shape = (
    temperature_factor
    * np.asarray(train_data["density"], dtype=float)**inversion_beta
)
training_mean_flux = float(train_data["intrinsic_flux"].mean())


def mean_flux_difference(log_amplitude):
    amplitude = np.exp(log_amplitude)
    return np.mean(np.exp(-amplitude * calibration_shape)) - training_mean_flux


A_fgpa = np.exp(brentq(mean_flux_difference, -20.0, 20.0))


def invert_flux_with_fgpa(flux, floor=flux_floor, ceiling=1.0-1.0e-5):
    '''Local FGPA inverse with explicit finite-flux limits.'''
    safe_flux = np.clip(flux, floor, ceiling)
    recovered_tau = -np.log(safe_flux)
    return (
        recovered_tau / (A_fgpa * temperature_factor)
    )**(1.0 / inversion_beta)


Delta_fgpa_test = invert_flux_with_fgpa(test_data["observed_flux"])
Delta_fgpa_intrinsic_test = invert_flux_with_fgpa(
    test_data["intrinsic_flux"], floor=1.0e-8, ceiling=1.0-1.0e-8
)

saturated_test = test_data["observed_flux"] <= flux_floor
Delta_saturation_lower_limit = invert_flux_with_fgpa(
    np.array([flux_floor])
)[0]

print(f"Training realistic-mock mean flux = {training_mean_flux:.4f}")
print(f"Effective FGPA amplitude A = {A_fgpa:.5f}")
print(f"Saturated test pixels = {saturated_test.sum()} / {saturated_test.size}")
print(f"FGPA reports only Delta_b >= {Delta_saturation_lower_limit:.2f} "
      "below the flux floor")

### Why direct FGPA loses saturated peaks

The local inverse maps every $F\leq F_{\rm floor}$ to the same number,

$$
\Delta_{\rm sat,min}=
\left[
\frac{-\ln F_{\rm floor}}
{A(T_0/10^4\,{\rm K})^{-0.7}}
\right]^{1/\beta}.
$$

This number is a lower limit, not a measurement of the peak height. FGPA also
ignores the non-local Voigt and instrumental mixing that generated the input.

## 8. Prepare neural-network tensors

Standardization is calculated from the training set only:

$$
\widetilde F=\frac{F-\mu_F}{\sigma_F},\qquad
\widetilde s=\frac{\ln\Delta_b-\mu_s}{\sigma_s}.
$$

PyTorch convolution expects tensors with shape
`[examples, channels, pixels]`. Here both the input and output have one channel.

In [ ]:
flux_mean = float(train_data["observed_flux"].mean())
flux_std = float(train_data["observed_flux"].std())

train_log_density = np.log(np.clip(train_data["density"], 1.0e-4, None))
validation_log_density = np.log(
    np.clip(validation_data["density"], 1.0e-4, None)
)
target_mean = float(train_log_density.mean())
target_std = float(train_log_density.std())


def flux_tensor(flux):
    standardized = (flux - flux_mean) / flux_std
    return torch.tensor(standardized[:, None, :], dtype=torch.float32)


def target_tensor(log_density):
    standardized = (log_density - target_mean) / target_std
    return torch.tensor(standardized[:, None, :], dtype=torch.float32)


training_set = TensorDataset(
    flux_tensor(train_data["observed_flux"]),
    target_tensor(train_log_density),
)
validation_set = TensorDataset(
    flux_tensor(validation_data["observed_flux"]),
    target_tensor(validation_log_density),
)

loader_generator = torch.Generator().manual_seed(7)
training_loader = DataLoader(
    training_set, batch_size=batch_size, shuffle=True,
    generator=loader_generator, num_workers=0
)
validation_loader = DataLoader(
    validation_set, batch_size=2*batch_size, shuffle=False, num_workers=0
)

example_flux, example_target = next(iter(training_loader))
print("Input batch shape: ", tuple(example_flux.shape))
print("Target batch shape:", tuple(example_target.shape))

## 9. Build the neural inverse gradually

The network maps a tensor of shape `[batch, 1, 128]` to another tensor of the same
shape. The single input channel is standardized flux; the single output channel is
standardized $\ln\Delta_b$.

We will assemble the model in four pieces:

1. a convolutional block;
2. an encoder and bottleneck;
3. a decoder without skip information;
4. the same decoder supplied with encoder skip features.

This makes the encoder–decoder and U-Net comparison controlled: their trainable
layers are identical, and only the information carried by the skips changes.

### 9.1 Convolutional block

A one-dimensional convolution slides a learnable kernel along the spectrum. For an odd kernel width $k=2r+1$, each output uses the central pixel and $r$ neighbours on either side:

$$
y_c(x)=\phi\!\left[
b_c+\sum_{c'}\sum_{j=-r}^{r}w_{cc'j}\,x_{c'}(x+j)
\right].
$$

The value of `convolution_kernel_size` in the configuration cell sets $k$ for every convolutional block. Two convolutions per block allow a richer nonlinear local mapping. Circular padding is appropriate for periodic simulation skewers; real spectra require boundary masks or non-periodic padding.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, input_channels, output_channels, kernel_size):
        super().__init__()
        if kernel_size < 1 or kernel_size % 2 == 0:
            raise ValueError("kernel_size must be a positive odd integer.")

        padding = kernel_size // 2
        self.layers = nn.Sequential(
            nn.Conv1d(
                input_channels, output_channels, kernel_size,
                padding=padding, padding_mode="reflect"
            ),
            nn.ReLU(),
            nn.Conv1d(
                output_channels, output_channels, kernel_size,
                padding=padding, padding_mode="reflect"
            ),
            nn.ReLU(),
        )

    def forward(self, inputs):
        return self.layers(inputs)


test_block = ConvBlock(
    input_channels=1,
    output_channels=base_channels,
    kernel_size=convolution_kernel_size,
)
with torch.no_grad():
    test_features = test_block(example_flux[:2])

print("Input to one convolutional block: ", tuple(example_flux[:2].shape))
print("Output from the block:            ", tuple(test_features.shape))


In [ ]:
# A small visual explanation of one configurable convolution.
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch


def labelled_box(ax, centre, text, colour, width=1.55, height=0.72):
    x_centre, y_centre = centre
    patch = FancyBboxPatch(
        (x_centre-width/2, y_centre-height/2), width, height,
        boxstyle="round,pad=0.04", facecolor=colour,
        edgecolor=BLACK, linewidth=1.0
    )
    ax.add_patch(patch)
    ax.text(x_centre, y_centre, text, ha="center", va="center", fontsize=10)


fig, ax = plt.subplots(figsize=(9.5, 2.6), constrained_layout=True)
labelled_box(
    ax,
    (1.0, 1.0),
    f"{convolution_kernel_size} neighbouring\nflux pixels",
    "aliceblue",
)
labelled_box(ax, (3.5, 1.0), "learned kernel\n+ bias", "bisque")
labelled_box(ax, (6.0, 1.0), "ReLU", "lavender", width=1.1)
labelled_box(
    ax, (8.5, 1.0), "one feature value\nat the central pixel", "honeydew"
)
for start, end in [(1.8, 2.7), (4.3, 5.3), (6.7, 7.6)]:
    ax.add_patch(FancyArrowPatch(
        (start, 1.0), (end, 1.0), arrowstyle="-|>",
        mutation_scale=13, color=GREY, linewidth=1.4
    ))
ax.set_xlim(0, 9.5)
ax.set_ylim(0.2, 1.8)
ax.axis("off")
ax.set_title("One-dimensional convolution at a single spectral position")
plt.show()


### 9.2 Encoder and bottleneck

The encoder extracts increasingly abstract features. After every encoder block, `MaxPool1d(2)` halves the number of spatial pixels, while the number of channels doubles. The value of `number_of_encoder_decoder_layers` controls how many of these block–pool stages are created. The bottleneck length is therefore

$$N_{\rm bottleneck}=128/2^{N_{\rm layers}}.$$

The encoder returns all intermediate feature maps as a list. The plain encoder–decoder will deliberately replace those maps by zeros; the U-Net will reuse the same list as skip features.


In [ ]:
class Encoder1D(nn.Module):
    def __init__(self, number_of_layers, kernel_size, base_channels=8):
        super().__init__()
        if number_of_layers < 1:
            raise ValueError("number_of_layers must be at least 1.")

        self.pool = nn.MaxPool1d(2)
        self.blocks = nn.ModuleList()

        input_channels = 1
        for layer_index in range(number_of_layers):
            output_channels = base_channels * 2**layer_index
            self.blocks.append(
                ConvBlock(input_channels, output_channels, kernel_size)
            )
            input_channels = output_channels

        bottleneck_channels = base_channels * 2**number_of_layers
        self.bottleneck = ConvBlock(
            input_channels, bottleneck_channels, kernel_size
        )

    def forward(self, inputs):
        features = []
        values = inputs

        for block in self.blocks:
            values = block(values)
            features.append(values)
            values = self.pool(values)

        latent = self.bottleneck(values)
        return latent, features


test_encoder = Encoder1D(
    number_of_layers=number_of_encoder_decoder_layers,
    kernel_size=convolution_kernel_size,
    base_channels=base_channels,
)
with torch.no_grad():
    latent, encoder_features = test_encoder(example_flux[:2])

print("input:       ", tuple(example_flux[:2].shape))
for layer_index, feature in enumerate(encoder_features, start=1):
    print(f"feature map {layer_index}:", tuple(feature.shape))
print("bottleneck:  ", tuple(latent.shape))


### 9.3 Decoder and the no-skip encoder–decoder

The decoder mirrors the number of encoder stages and upsamples the bottleneck back to 128 pixels. The kernel size is shared by all encoder and decoder blocks. To keep the two neural models exactly depth- and layer-matched, the decoder always reserves channel slots for every skip feature. In the plain encoder–decoder those slots are filled with zeros, so it must reconstruct the field using only the bottleneck.

This is a supervised inverse network with an autoencoder-like compress–expand geometry. It is not a classical autoencoder, because its target is density rather than a reconstruction of its flux input.


In [ ]:
class Decoder1D(nn.Module):
    def __init__(self, number_of_layers, kernel_size, base_channels=8):
        super().__init__()
        self.blocks = nn.ModuleList()

        current_channels = base_channels * 2**number_of_layers
        for layer_index in reversed(range(number_of_layers)):
            skip_channels = base_channels * 2**layer_index
            self.blocks.append(
                ConvBlock(
                    current_channels + skip_channels,
                    skip_channels,
                    kernel_size,
                )
            )
            current_channels = skip_channels

        self.output = nn.Conv1d(
            current_channels, 1, kernel_size=1
        )

    def forward(self, latent, encoder_features, use_skips):
        decoded = latent

        for block, feature in zip(
            self.blocks, reversed(encoder_features)
        ):
            skip_feature = feature if use_skips else torch.zeros_like(feature)
            decoded = F.interpolate(
                decoded,
                size=feature.shape[-1],
                mode="linear",
                align_corners=False,
            )
            decoded = block(
                torch.cat([decoded, skip_feature], dim=1)
            )

        return self.output(decoded)


class InversionNetwork1D(nn.Module):
    def __init__(
        self,
        use_skips,
        number_of_layers,
        kernel_size,
        base_channels=8,
    ):
        super().__init__()
        self.use_skips = use_skips
        self.encoder = Encoder1D(
            number_of_layers=number_of_layers,
            kernel_size=kernel_size,
            base_channels=base_channels,
        )
        self.decoder = Decoder1D(
            number_of_layers=number_of_layers,
            kernel_size=kernel_size,
            base_channels=base_channels,
        )

    def forward(self, inputs):
        latent, encoder_features = self.encoder(inputs)
        return self.decoder(
            latent,
            encoder_features,
            use_skips=self.use_skips,
        )


def parameter_count(model):
    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


torch.manual_seed(11)
encoder_decoder = InversionNetwork1D(
    use_skips=False,
    number_of_layers=number_of_encoder_decoder_layers,
    kernel_size=convolution_kernel_size,
    base_channels=base_channels,
).to(device)
initial_weights = {
    name: value.detach().cpu().clone()
    for name, value in encoder_decoder.state_dict().items()
}

with torch.no_grad():
    plain_output = encoder_decoder(example_flux[:2].to(device))

print("Encoder-decoder output:", tuple(plain_output.shape))
print(f"Trainable parameters: {parameter_count(encoder_decoder):,}")


In [ ]:
def draw_network_schematic(ax, use_skips, title):
    input_length = example_flux.shape[-1]

    keys = ["input"]
    labels = {"input": f"Flux\n{input_length} x 1"}
    colours = {"input": "aliceblue"}

    for layer_index in range(number_of_encoder_decoder_layers):
        key = f"enc_{layer_index}"
        channels = base_channels * 2**layer_index
        length = input_length // 2**layer_index
        keys.append(key)
        labels[key] = f"Encoder {layer_index + 1}\n{length} x {channels}"
        colours[key] = "aliceblue" if layer_index % 2 == 0 else "lightblue"

    bottleneck_length = input_length // 2**number_of_encoder_decoder_layers
    bottleneck_channels = base_channels * 2**number_of_encoder_decoder_layers
    keys.append("latent")
    labels["latent"] = (
        f"Bottleneck\n{bottleneck_length} x {bottleneck_channels}"
    )
    colours["latent"] = "lavender"

    for layer_index in reversed(range(number_of_encoder_decoder_layers)):
        key = f"dec_{layer_index}"
        channels = base_channels * 2**layer_index
        length = input_length // 2**layer_index
        keys.append(key)
        labels[key] = f"Decoder {layer_index + 1}\n{length} x {channels}"
        colours[key] = "bisque" if layer_index % 2 else "blanchedalmond"

    keys.append("output")
    labels["output"] = f"ln density\n{input_length} x 1"
    colours["output"] = "honeydew"

    node_spacing = 1.8
    positions = {
        key: (0.75 + node_spacing * index, 1.0)
        for index, key in enumerate(keys)
    }

    for key in keys:
        labelled_box(
            ax, positions[key], labels[key], colours[key],
            width=1.38, height=0.68
        )
    for left, right in zip(keys[:-1], keys[1:]):
        ax.add_patch(FancyArrowPatch(
            (positions[left][0] + 0.70, 1.0),
            (positions[right][0] - 0.70, 1.0),
            arrowstyle="-|>", mutation_scale=11,
            color=GREY, linewidth=1.2
        ))

    if use_skips:
        for layer_index in range(number_of_encoder_decoder_layers):
            source = f"enc_{layer_index}"
            target = f"dec_{layer_index}"
            arc_radius = 0.12 + 0.035 * (
                number_of_encoder_decoder_layers - layer_index
            )
            ax.add_patch(FancyArrowPatch(
                (positions[source][0], 1.38),
                (positions[target][0], 1.38),
                connectionstyle=f"arc3,rad={arc_radius}",
                arrowstyle="-|>", mutation_scale=11,
                color=RED, linewidth=1.8
            ))
        ax.text(
            positions["latent"][0], 2.05,
            "skip connections", color=RED,
            ha="center", fontweight="bold"
        )
    else:
        ax.text(
            positions["latent"][0], 1.90,
            "all information passes through the bottleneck",
            color=PURPLE, ha="center", fontweight="bold"
        )

    ax.set_xlim(0, positions["output"][0] + 0.75)
    ax.set_ylim(0.35, 2.35)
    ax.axis("off")
    ax.set_title(title, loc="left", fontweight="bold")


schematic_width = max(
    13.5,
    1.8 * (2 * number_of_encoder_decoder_layers + 3),
)
fig, axes = plt.subplots(
    2, 1, figsize=(schematic_width, 5.3), constrained_layout=True
)
draw_network_schematic(
    axes[0], use_skips=False,
    title="A. Depth-matched encoder-decoder: skip channels contain zeros"
)
draw_network_schematic(
    axes[1], use_skips=True,
    title="B. 1D U-Net: encoder features are copied to the decoder"
)
plt.show()


## 10. Training procedure

Both networks minimize mean-squared error in standardized log density,

$$
\mathcal L=\frac{1}{N_{\rm batch}N_{\rm pix}}
\sum(\widetilde s_{\rm pred}-\widetilde s_{\rm true})^2.
$$

Validation examples receive no gradient updates. The state with the lowest
validation loss is retained. The two models begin from identical trainable weights
and receive the same deterministic mini-batch ordering.

In [ ]:
def make_training_loader(seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        training_set, batch_size=batch_size, shuffle=True,
        generator=generator, num_workers=0
    )


def train_model(model, loader_seed):
    training_loader_local = make_training_loader(loader_seed)
    loss_function = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    history = {"training": [], "validation": []}
    best_validation_loss = np.inf
    best_state = None
    start = time.perf_counter()

    for epoch in range(1, number_of_epochs + 1):
        model.train()
        training_sum = 0.0
        for input_batch, target_batch in training_loader_local:
            input_batch = input_batch.to(device)
            target_batch = target_batch.to(device)

            optimizer.zero_grad()
            prediction = model(input_batch)
            loss = loss_function(prediction, target_batch)
            loss.backward()
            optimizer.step()
            training_sum += loss.item() * input_batch.size(0)

        model.eval()
        validation_sum = 0.0
        with torch.no_grad():
            for input_batch, target_batch in validation_loader:
                input_batch = input_batch.to(device)
                target_batch = target_batch.to(device)
                validation_sum += (
                    loss_function(model(input_batch), target_batch).item()
                    * input_batch.size(0)
                )

        training_loss = training_sum / len(training_set)
        validation_loss = validation_sum / len(validation_set)
        history["training"].append(training_loss)
        history["validation"].append(validation_loss)

        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in model.state_dict().items()
            }

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"epoch {epoch:2d}/{number_of_epochs}: "
                f"train={training_loss:.4f}, validation={validation_loss:.4f}"
            )

    model.load_state_dict(best_state)
    print(f"Retained best model; CPU time={time.perf_counter()-start:.1f} s")
    return history

### 10.1 Train the encoder–decoder first

This establishes what can be learned when every reconstruction feature must pass
through the 32-pixel bottleneck. Watch whether its validation curve follows its
training curve or begins to separate.

In [ ]:
print("Training the no-skip encoder-decoder")
history_encoder_decoder = train_model(encoder_decoder, loader_seed=29)

### 10.2 Add the U-Net skip connections

We now instantiate the same trainable layers and restore exactly the same initial
weights. The only functional change is `use_skips=True`: the 128- and 64-pixel
encoder features are concatenated into the corresponding decoder stages.

Skip connections help preserve the alignment of narrow absorption features that
may be blurred when everything is compressed through the bottleneck.

After training, the best-validation U-Net and its normalization are exported as a portable inference model for the companion DDPM notebook.


In [ ]:
unet = InversionNetwork1D(
    use_skips=True,
    number_of_layers=number_of_encoder_decoder_layers,
    kernel_size=convolution_kernel_size,
    base_channels=base_channels,
).to(device)
unet.load_state_dict(initial_weights)

print(f"Encoder-decoder parameters: {parameter_count(encoder_decoder):,}")
print(f"U-Net parameters:          {parameter_count(unet):,}")
assert parameter_count(encoder_decoder) == parameter_count(unet)

print("\nTraining the U-Net")
history_unet = train_model(unet, loader_seed=29)


class SavedUNetDensityPredictor(nn.Module):
    '''Bundle the trained U-Net and its normalization for inference.'''
    def __init__(self, network):
        super().__init__()
        self.network = network
        self.register_buffer(
            "flux_mean", torch.tensor(flux_mean, dtype=torch.float32)
        )
        self.register_buffer(
            "flux_std", torch.tensor(flux_std, dtype=torch.float32)
        )
        self.register_buffer(
            "target_mean", torch.tensor(target_mean, dtype=torch.float32)
        )
        self.register_buffer(
            "target_std", torch.tensor(target_std, dtype=torch.float32)
        )

    def forward(self, observed_flux):
        standardized_flux = (observed_flux - self.flux_mean) / self.flux_std
        standardized_log_density = self.network(
            standardized_flux[:, None, :]
        )[:, 0, :]
        log_density = (
            self.target_mean + self.target_std * standardized_log_density
        )
        return torch.exp(torch.clamp(
            log_density, min=float(np.log(1.0e-4)), max=float(np.log(1.0e3))
        ))


unet.eval()
saved_predictor = SavedUNetDensityPredictor(unet).to(device).eval()
trace_example = torch.as_tensor(
    validation_data["observed_flux"][:2], dtype=torch.float32, device=device
)
traced_predictor = torch.jit.trace(saved_predictor, trace_example)
unet_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
torch.jit.save(traced_predictor, str(unet_checkpoint_path))
print(f"Saved trained U-Net predictor to {unet_checkpoint_path}")

In [ ]:
epochs = np.arange(1, number_of_epochs + 1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)

for ax, history, title, colour in [
    (axes[0], history_encoder_decoder, "Encoder-decoder", PURPLE),
    (axes[1], history_unet, "U-Net", ORANGE),
]:
    ax.semilogy(epochs, history["training"], color=BLUE, lw=1.8,
                label="training")
    ax.semilogy(epochs, history["validation"], color=colour, lw=2.1,
                label="validation")
    best_epoch = int(np.argmin(history["validation"])) + 1
    ax.axvline(best_epoch, color=GREY, ls=":",
               label=f"selected epoch {best_epoch}")
    ax.set(xlabel="epoch", ylabel="mean-squared error", title=title)
    ax.grid(alpha=0.2)
    ax.legend()
plt.show()

## 11. Predict the untouched test simulations

Network outputs are transformed from standardized log density back to physical
overdensity:

$$
\Delta_b^{\rm NN}
=\exp(\mu_s+\sigma_s\widetilde s_{\rm pred}).
$$

The numerical output limits are much wider than the diffuse-IGM regime and only
prevent overflow from pathological predictions.

In [ ]:
def predict_density(model, observed_flux):
    inputs = flux_tensor(observed_flux).to(device)
    model.eval()
    with torch.no_grad():
        standardized = model(inputs).cpu().numpy()[:, 0, :]

    log_prediction = target_mean + target_std * standardized
    log_prediction = np.clip(log_prediction, np.log(1.0e-4), np.log(1.0e3))
    return np.exp(log_prediction)


Delta_encoder_decoder_test = predict_density(
    encoder_decoder, test_data["observed_flux"]
)
Delta_unet_test = predict_density(unet, test_data["observed_flux"])

print("Test predictions:", Delta_unet_test.shape)

## 12. Field-level comparison on one representative sightline

The upper panel is the shared input. The lower panel compares true gas density,
direct FGPA, the no-skip encoder–decoder, and the U-Net. This plot tests whether
structures occur at the correct positions; summary statistics alone cannot do so.

In [ ]:
truth_example = test_data["density"][representative]
flux_example = test_data["observed_flux"][representative]

fig, axes = plt.subplots(
    2, 1, figsize=(11, 7.0), sharex=True, constrained_layout=True
)
axes[0].step(x, flux_example, where="mid", color=BLACK, lw=1.0,
             label="observed noisy spectrum")
axes[0].plot(x, test_data["instrumental_flux"][representative],
             color=BLUE, lw=1.7, alpha=0.8, label="noise-free instrument flux")
axes[0].set_ylim(-0.12, 1.12)
axes[0].set_ylabel("transmitted flux")
axes[0].set_title(f"Held-out simulation {representative_simulation}")
axes[0].legend(ncol=2)

axes[1].semilogy(x, truth_example, color=BLACK, lw=2.5, label="truth")
axes[1].semilogy(x, Delta_fgpa_test[representative], color=BLUE, lw=1.5,
                 label="direct FGPA")
axes[1].semilogy(x, Delta_encoder_decoder_test[representative],
                 color=PURPLE, lw=1.8, label="encoder-decoder")
axes[1].semilogy(x, Delta_unet_test[representative],
                 color=ORANGE, lw=2.1, label="U-Net")
axes[1].axhline(1.0, color=GREY, ls=":", lw=0.8)
axes[1].set_ylabel(r"gas overdensity $\Delta_b$")
axes[1].set_xlabel(r"distance $x$ [$h^{-1}$ cMpc]")
axes[1].legend(ncol=2)
plt.show()

## 13. Matched-resolution field metrics

We smooth truth and all predictions to a common $1\,h^{-1}{\rm cMpc}$ FWHM before
calculating primary field metrics. Within each of simulations 20–26, metrics are
averaged over its 16 test sightlines. The quoted uncertainty is then the standard
deviation between the seven simulation-level values.

This preserves the simulation—not the individual sightline—as the independent
unit of the error bar.

In [ ]:
comparison_sigma_pixels = (
    comparison_fwhm
    / (2.0 * np.sqrt(2.0 * np.log(2.0)))
    / cell_size
)


def smooth_fields(fields):
    return ndimage.gaussian_filter1d(
        fields, comparison_sigma_pixels, axis=1, mode="wrap"
    )


def field_metrics_per_skewer(truth, prediction):
    log_truth = np.log10(np.clip(truth, 1.0e-4, None))
    log_prediction = np.log10(np.clip(prediction, 1.0e-4, None))
    difference = log_prediction - log_truth
    return {
        "RMSE [dex]": np.sqrt(np.mean(difference**2, axis=1)),
        "bias [dex]": np.mean(difference, axis=1),
        "correlation": np.array([
            np.corrcoef(true_row, predicted_row)[0, 1]
            for true_row, predicted_row in zip(log_truth, log_prediction)
        ]),
    }


def average_within_each_test_simulation(values):
    return np.array([
        np.mean(values[test_data["simulation"] == simulation], axis=0)
        for simulation in test_simulations
    ])


truth_matched = smooth_fields(test_data["density"])
fgpa_matched = smooth_fields(Delta_fgpa_test)
encoder_decoder_matched = smooth_fields(Delta_encoder_decoder_test)
unet_matched = smooth_fields(Delta_unet_test)

matched_predictions = {
    "Direct FGPA": fgpa_matched,
    "Encoder-decoder": encoder_decoder_matched,
    "U-Net": unet_matched,
}

metric_samples = {}
for method, prediction in matched_predictions.items():
    per_skewer = field_metrics_per_skewer(truth_matched, prediction)
    metric_samples[method] = {
        metric: average_within_each_test_simulation(values)
        for metric, values in per_skewer.items()
    }

print("Mean +/- simulation-to-simulation standard deviation")
print(f"{'Method':18s} {'RMSE [dex]':>20s} {'bias [dex]':>20s} "
      f"{'correlation':>20s}")
for method, samples in metric_samples.items():
    print(
        f"{method:18s} "
        f"{samples['RMSE [dex]'].mean():7.3f} +/- "
        f"{samples['RMSE [dex]'].std(ddof=1):.3f} "
        f"{samples['bias [dex]'].mean():+7.3f} +/- "
        f"{samples['bias [dex]'].std(ddof=1):.3f} "
        f"{samples['correlation'].mean():7.3f} +/- "
        f"{samples['correlation'].std(ddof=1):.3f}"
    )

In [ ]:
method_colours = {
    "Direct FGPA": BLUE,
    "Encoder-decoder": PURPLE,
    "U-Net": ORANGE,
}

methods = list(metric_samples)
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.2), constrained_layout=True)

rmse_mean = [metric_samples[m]["RMSE [dex]"].mean() for m in methods]
rmse_std = [metric_samples[m]["RMSE [dex]"].std(ddof=1) for m in methods]
correlation_mean = [metric_samples[m]["correlation"].mean() for m in methods]
correlation_std = [metric_samples[m]["correlation"].std(ddof=1) for m in methods]

axes[0].bar(methods, rmse_mean, yerr=rmse_std, capsize=4,
            color=[method_colours[m] for m in methods])
axes[0].set_ylabel("log-density RMSE [dex]")
axes[0].set_title("Lower is better")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=0.2)

axes[1].bar(methods, correlation_mean, yerr=correlation_std, capsize=4,
            color=[method_colours[m] for m in methods])
axes[1].set_ylim(max(0.0, min(correlation_mean)-0.15), 1.0)
axes[1].set_ylabel("log-density correlation")
axes[1].set_title("Higher is better")
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=0.2)

fig.suptitle("Field accuracy across seven held-out simulations")
plt.show()

## 14. Summary statistics: PDF, power spectrum, and bispectrum

We now ask whether the reconstructed fields reproduce progressively richer
statistics.

### Density PDF

The PDF of $\log_{10}\Delta_b$ measures the one-point distribution but discards
spatial ordering.

### One-dimensional power spectrum

For $\delta_b=\Delta_b/\langle\Delta_b\rangle-1$,

$$P_{1\rm D}(k)=\frac{|\widetilde\delta_b(k)|^2}{L}.$$

It measures scale-dependent variance but not Fourier-phase coupling.

### One-dimensional folded bispectrum

A one-dimensional closed Fourier triangle satisfies $k_1+k_2+k_3=0$. We show the
folded/diagonal configuration $(k,k,-2k)$:

$$B_{1\rm D}(k,k,-2k)
=\frac{1}{L}\,\mathrm{Re}\!\left[
\widetilde\delta_b(k)^2\widetilde\delta_b^*(2k)
\right].$$

This probes non-Gaussian phase coupling that is absent from the power spectrum.

In [ ]:
pdf_edges = np.linspace(-2.0, 2.0, 33)
pdf_centres = 0.5 * (pdf_edges[:-1] + pdf_edges[1:])


def density_pdf_per_skewer(density):
    return np.asarray([
        np.histogram(np.log10(np.clip(row, 1.0e-4, None)),
                     bins=pdf_edges, density=True)[0]
        for row in density
    ])


def power_per_skewer(density):
    contrast = density / density.mean(axis=1, keepdims=True) - 1.0
    transform = cell_size * np.fft.rfft(contrast, axis=1)
    k = 2.0 * np.pi * np.fft.rfftfreq(number_of_cells, d=cell_size)
    power = np.abs(transform)**2 / box_size
    return k[1:], power[:, 1:]


def folded_bispectrum_per_skewer(density):
    contrast = density / density.mean(axis=1, keepdims=True) - 1.0
    transform = cell_size * np.fft.rfft(contrast, axis=1)
    modes = np.arange(1, number_of_cells // 4)
    k = 2.0 * np.pi * modes / box_size
    bispectrum = np.real(
        transform[:, modes]**2 * np.conj(transform[:, 2*modes])
    ) / box_size
    return k, bispectrum


def bin_modes(k, statistic, number_of_bins):
    edges = np.logspace(np.log10(k.min()), np.log10(k.max()),
                        number_of_bins + 1)
    centres = np.sqrt(edges[:-1] * edges[1:])
    binned = np.full((statistic.shape[0], number_of_bins), np.nan)
    for index in range(number_of_bins):
        in_bin = (k >= edges[index]) & (k < edges[index+1])
        if np.any(in_bin):
            binned[:, index] = statistic[:, in_bin].mean(axis=1)
    return centres, binned

In [ ]:
evaluation_fields = {
    "Truth": truth_matched,
    "Direct FGPA": fgpa_matched,
    "Encoder-decoder": encoder_decoder_matched,
    "U-Net": unet_matched,
}

summary_statistics = {}
for method, fields in evaluation_fields.items():
    pdf_skewers = density_pdf_per_skewer(fields)

    power_k_raw, power_raw = power_per_skewer(fields)
    power_k, power_skewers = bin_modes(power_k_raw, power_raw, 10)

    bispectrum_k_raw, bispectrum_raw = folded_bispectrum_per_skewer(fields)
    bispectrum_k, bispectrum_skewers = bin_modes(
        bispectrum_k_raw, bispectrum_raw, 8
    )

    summary_statistics[method] = {
        "PDF": average_within_each_test_simulation(pdf_skewers),
        "power": average_within_each_test_simulation(power_skewers),
        "bispectrum": average_within_each_test_simulation(bispectrum_skewers),
    }

print("Each summary array has first dimension = seven held-out simulations")
for method, statistics in summary_statistics.items():
    print(method, {name: values.shape for name, values in statistics.items()})

In [ ]:
summary_colours = {
    "Truth": BLACK,
    "Direct FGPA": BLUE,
    "Encoder-decoder": PURPLE,
    "U-Net": ORANGE,
}
summary_markers = {
    "Truth": "o", "Direct FGPA": "s",
    "Encoder-decoder": "D", "U-Net": "^",
}

fig, axes = plt.subplots(3, figsize=(5, 16), constrained_layout=True)

for method, statistics in summary_statistics.items():
    colour = summary_colours[method]
    marker = summary_markers[method]

    pdf_mean = np.nanmean(statistics["PDF"], axis=0)
    pdf_std = np.nanstd(statistics["PDF"], axis=0, ddof=1)
    valid_pdf = np.isfinite(pdf_mean) & (pdf_mean > 0.0)
    axes[0].errorbar(
        pdf_centres[valid_pdf], pdf_mean[valid_pdf], yerr=pdf_std[valid_pdf],
        color=colour, marker=marker, ms=3.5, lw=1.4,
        capsize=1.8, errorevery=2, label=method
    )

    power_mean = np.nanmean(statistics["power"], axis=0)
    power_std = np.nanstd(statistics["power"], axis=0, ddof=1)
    valid_power = np.isfinite(power_mean)
    axes[1].errorbar(
        power_k[valid_power], power_mean[valid_power],
        yerr=power_std[valid_power], color=colour,
        marker=marker, ms=4, lw=1.4, capsize=2, label=method
    )

    bispectrum_mean = np.nanmean(statistics["bispectrum"], axis=0)
    bispectrum_std = np.nanstd(statistics["bispectrum"], axis=0, ddof=1)
    valid_bispectrum = np.isfinite(bispectrum_mean)
    axes[2].errorbar(
        bispectrum_k[valid_bispectrum],
        bispectrum_mean[valid_bispectrum],
        yerr=bispectrum_std[valid_bispectrum],
        color=colour, marker=marker, ms=4,
        lw=1.4, capsize=2, label=method
    )

axes[0].set(
    xlabel=r"$\log_{10}\Delta_b$", ylabel="probability density",
    title="Density PDF"
)

axes[1].set(
    xlabel=r"$k$ [$h$ cMpc$^{-1}$]",
    ylabel=r"$P_{1\rm D}(k)$ [$h^{-1}$ cMpc]",
    title="1D power spectrum"
)
axes[1].set_xscale("log")
axes[1].set_yscale("log")

all_bispectrum_means = np.concatenate([
    np.nanmean(statistics["bispectrum"], axis=0)
    for statistics in summary_statistics.values()
])
linear_threshold = max(
    1.0e-8, 0.03*np.nanmax(np.abs(all_bispectrum_means))
)
axes[2].set(
    xlabel=r"$k$ [$h$ cMpc$^{-1}$]",
    ylabel=r"$B_{1\rm D}(k,k,-2k)$ [$(h^{-1}$ cMpc$)^2$]",
    title="Folded 1D bispectrum"
)
axes[2].set_xscale("log")
axes[2].set_yscale("log")
axes[2].axhline(0.0, color=GREY, lw=0.7)

for ax in axes:
    ax.grid(alpha=0.18, which="both")
axes[0].legend(fontsize=8)
fig.suptitle(
    "Mean and standard deviation across held-out simulations 20–26",
    fontsize=15
)
plt.show()